# 01 — Acquisition & nettoyage des données réelles
## Prix des céréales & légumineuses au Sénégal — WFP & Banque mondiale (2007–2026)

**100 % données réelles et officielles :**
- 🥗 **WFP / HDX** — prix de marché *réels* (relevés `actual`, *Retail*, KG),
  mensuels, **64 marchés**, **14 régions**, depuis 2000.
  → [data.humdata.org/dataset/wfp-food-prices-for-senegal](https://data.humdata.org/dataset/wfp-food-prices-for-senegal)
- 🌍 **Banque mondiale (API)** — inflation officielle, PIB/hab., population,
  production alimentaire, etc.
- 🗺️ **geoBoundaries** — contours des 14 régions.

> ⚙️ Les fichiers sont d'abord téléchargés par `scripts/download_data.py`.
> Ce notebook les nettoie et produit un modèle en étoile dans `data/processed/`.


In [1]:

import os, warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 110, "axes.titlesize": 13})

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw")
PROC = os.path.join(PROJ, "data", "processed")
GEO = os.path.join(PROJ, "data", "geo")
FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS):
    os.makedirs(d, exist_ok=True)

# Libellés FR des denrées et des régions
COMMOD_FR = {
    "Rice (imported)": "Riz importé (brisé)", "Rice (local)": "Riz local",
    "Rice (ordinary, first quality)": "Riz ordinaire 1re qual.",
    "Rice (ordinary, second quality)": "Riz ordinaire 2e qual.",
    "Millet": "Mil", "Sorghum": "Sorgho", "Sorghum (imported)": "Sorgho importé",
    "Maize (local)": "Maïs local", "Maize (imported)": "Maïs importé",
    "Beans (niebe)": "Niébé (haricot)", "Groundnuts (shelled)": "Arachide décortiquée",
    "Groundnuts (unshelled)": "Arachide en coque",
}
REGION_FR = {"Saint Louis": "Saint-Louis", "Thies": "Thiès",
             "Kedougou": "Kédougou", "Sedhiou": "Sédhiou"}
print("Racine projet :", PROJ)


Racine projet : C:\projet\senegal-food-prices


### 1. Chargement des données brutes (la 2ᵉ ligne du CSV WFP contient les balises HXL → ignorée)

In [2]:

prix = pd.read_csv(os.path.join(RAW, "wfp_food_prices_sen.csv"), skiprows=[1])
marches = pd.read_csv(os.path.join(RAW, "wfp_markets_sen.csv"), skiprows=[1])
wb = pd.read_csv(os.path.join(RAW, "worldbank_senegal.csv"))
print("Prix bruts :", prix.shape)
print("Période    :", prix["date"].min(), "->", prix["date"].max())
print("Colonnes   :", list(prix.columns))
prix.head(3)


Prix bruts : (50059, 16)
Période    : 2000-01-15 -> 2026-03-15
Colonnes   : ['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude', 'longitude', 'category', 'commodity', 'commodity_id', 'unit', 'priceflag', 'pricetype', 'currency', 'price', 'usdprice']


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2000-01-15,Dakar,Dakar,Tilene,447,14.68,-17.46,cereals and tubers,Millet,73,KG,actual,Retail,XOF,124.75,0.30
1,2000-01-15,Dakar,Dakar,Tilene,447,14.68,-17.46,cereals and tubers,Rice (imported),64,KG,actual,Retail,XOF,227.50,0.54
2,2000-01-15,Dakar,Dakar,Tilene,447,14.68,-17.46,cereals and tubers,Sorghum,65,KG,actual,Retail,XOF,125.00,0.30


### 2. Diagnostic qualité

In [3]:

print("Valeurs manquantes :\n", prix[["date","admin1","commodity","price"]].isna().sum().to_dict())
print("\npriceflag :", prix["priceflag"].value_counts().to_dict())
print("pricetype :", prix["pricetype"].value_counts().to_dict())
print("unités    :", prix["unit"].value_counts().to_dict())
print("devises   :", prix["currency"].value_counts().to_dict())
print("\nRelevés par an :")
print(prix.assign(an=pd.to_datetime(prix['date']).dt.year).groupby("an").size().loc[[2000,2006,2007,2015,2024,2026]].to_dict())


Valeurs manquantes :
 {'date': 0, 'admin1': 0, 'commodity': 0, 'price': 0}

priceflag : {'actual': 50059}
pricetype : {'Retail': 50059}
unités    : {'KG': 50059}
devises   : {'XOF': 50059}

Relevés par an :
{2000: 38, 2006: 42, 2007: 2273, 2015: 1996, 2024: 4567, 2026: 1042}


### 3. Nettoyage
- Conversion des dates au 1er du mois.
- Filtres : `actual`, *Retail*, unité KG, prix > 0.
- Libellés FR ; on **restreint à partir de 2007** (couverture fiable : ~52 marchés).
- Suppression des doublons.

In [4]:

p = prix.copy()
p["date"] = pd.to_datetime(p["date"]).values.astype("datetime64[M]")
p = p[(p["priceflag"] == "actual") & (p["pricetype"] == "Retail") & (p["unit"] == "KG")]
p = p[p["price"] > 0].dropna(subset=["price", "commodity", "admin1"])
p["commodity_fr"] = p["commodity"].map(COMMOD_FR).fillna(p["commodity"])
p["region"] = p["admin1"]                       # nom brut (jointure GeoJSON)
p["region_fr"] = p["region"].replace(REGION_FR) # nom d'affichage
p = p[p["date"] >= "2007-01-01"]
n0 = len(p)
p = p.drop_duplicates(subset=["date", "market", "commodity_fr"])
print(f"Lignes retenues : {len(p):,} (doublons retirés : {n0-len(p):,})")
print("Denrées :", sorted(p["commodity_fr"].unique()))
print("Régions :", sorted(p["region_fr"].unique()))


Lignes retenues : 49,771 (doublons retirés : 0)
Denrées : ['Arachide décortiquée', 'Arachide en coque', 'Maïs importé', 'Maïs local', 'Mil', 'Niébé (haricot)', 'Riz importé (brisé)', 'Riz local', 'Riz ordinaire 1re qual.', 'Riz ordinaire 2e qual.', 'Sorgho', 'Sorgho importé']
Régions : ['Dakar', 'Diourbel', 'Fatick', 'Kaffrine', 'Kaolack', 'Kolda', 'Kédougou', 'Louga', 'Matam', 'Saint-Louis', 'Sédhiou', 'Tambacounda', 'Thiès', 'Ziguinchor']


### 4. Agrégation : prix national & régional (médiane robuste entre marchés)

In [5]:

# National : médiane des marchés par mois et denrée
fact_nat = (p.groupby(["date", "commodity_fr"])
              .agg(prix_median=("price", "median"),
                   prix_moyen=("price", "mean"),
                   nb_marches=("market", "nunique"))
              .reset_index())
# Régional
fact_reg = (p.groupby(["date", "region", "region_fr", "commodity_fr"])
              .agg(prix_median=("price", "median"),
                   nb_marches=("market", "nunique"))
              .reset_index())
# Lissage des valeurs aberrantes isolées (médiane glissante robuste / MAD)
def winsorize_group(g):
    med = g.rolling(13, center=True, min_periods=4).median()
    mad = (g - med).abs().rolling(13, center=True, min_periods=4).median()
    out = g.copy(); m = (g - med).abs() > 3.5 * mad.replace(0, np.nan)
    out[m] = med[m]; return out
fact_nat = fact_nat.sort_values(["commodity_fr", "date"])
n_out = 0
for c, gg in fact_nat.groupby("commodity_fr"):
    w_ = winsorize_group(gg["prix_median"]); n_out += int((w_ != gg["prix_median"]).sum())
    fact_nat.loc[gg.index, "prix_median"] = w_.values
print(f"fact national : {fact_nat.shape} | fact régional : {fact_reg.shape} "
      f"| points aberrants lissés : {n_out}")
fact_nat.tail(3)


fact national : (1655, 5) | fact régional : (18805, 6) | points aberrants lissés : 124

,date,commodity_fr,prix_median,prix_moyen,nb_marches
1632,2026-01-01,Sorgho importé,300.0,302.500000,16
1643,2026-02-01,Sorgho importé,300.0,318.764706,17
1654,2026-03-01,Sorgho importé,300.0,303.750000,12


### 5. Indice du panier céréalier (base 100 = 2015)
Panier pondéré reflétant le régime alimentaire sénégalais :
**riz importé 40 %, mil 25 %, maïs local 15 %, riz local 10 %, sorgho 10 %**.
Indice = moyenne pondérée des prix relatifs (prix / prix moyen de 2015).

In [6]:

PANIER = {"Riz importé (brisé)": 0.40, "Mil": 0.25, "Maïs local": 0.15,
          "Riz local": 0.10, "Sorgho": 0.10}
piv = (fact_nat[fact_nat["commodity_fr"].isin(PANIER)]
       .pivot(index="date", columns="commodity_fr", values="prix_median")
       .asfreq("MS").interpolate(limit=3))
base = piv.loc["2015"].mean()                      # prix moyen 2015 par denrée
rel = piv.divide(base, axis=1) * 100               # prix relatifs (base 100=2015)
w = pd.Series(PANIER)
indice = (rel[w.index] * w).sum(axis=1) / w.sum()
panier_nat = pd.DataFrame({"date": indice.index, "indice_panier": indice.values})
panier_nat["var_annuelle_pct"] = panier_nat["indice_panier"].pct_change(12) * 100
print("Indice panier — base 2015≈100 :", round(panier_nat.set_index('date').loc['2015','indice_panier'].mean(),1))
print("Dernier indice :", round(panier_nat['indice_panier'].iloc[-1],1),
      "(", panier_nat['date'].iloc[-1].strftime('%b %Y'), ")")
panier_nat.tail(3)


Indice panier — base 2015≈100 : 100.0
Dernier indice : 127.8 ( Mar 2026 )


,date,indice_panier,var_annuelle_pct
228,2026-01-01,128.698361,-10.770985
229,2026-02-01,129.130415,-11.524807
230,2026-03-01,127.842383,-11.032749


### 6. Inflation alimentaire WFP vs inflation officielle (Banque mondiale)

In [7]:

panier_nat["annee"] = panier_nat["date"].dt.year
wfp_an = panier_nat.dropna(subset=["var_annuelle_pct"]).groupby("annee")["var_annuelle_pct"].mean()
wb_infl = wb[wb["code"] == "FP.CPI.TOTL.ZG"].set_index("annee")["valeur"]
comp = pd.DataFrame({"inflation_alimentaire_WFP_%": wfp_an.round(1),
                     "inflation_officielle_BM_%": wb_infl.round(1)}).dropna()
print(comp.loc[2008:].to_string())
print("\nCorrélation WFP vs officielle :",
      round(comp["inflation_alimentaire_WFP_%"].corr(comp["inflation_officielle_BM_%"]), 2))


       inflation_alimentaire_WFP_%  inflation_officielle_BM_%
annee                                                        
2008                          24.8                        7.3
2009                          -4.3                       -2.2
2010                          -5.5                        1.2
2011                           5.3                        3.4
2012                           2.7                        1.4
2013                           3.2                        0.7
2014                          -3.2                       -1.1
2015                          -0.6                        0.1
2016                          -1.5                        0.8
2017                           8.2                        1.3
2018                          -1.6                        0.5
2019                           3.2                        1.8
2020                           8.0                        2.5
2021                           0.7                        2.2
2022    

### 7. Prix réels (déflatés par l'IPC officiel) + indicateurs macro

In [8]:

cpi = wb[wb["code"] == "FP.CPI.TOTL"][["annee", "valeur"]].rename(columns={"valeur": "cpi"})
fact_nat["annee"] = fact_nat["date"].dt.year
fact_nat = fact_nat.merge(cpi, on="annee", how="left")
fact_nat["cpi"] = fact_nat["cpi"].ffill().bfill()
fact_nat["prix_reel_2010"] = fact_nat["prix_median"] / fact_nat["cpi"] * 100
# Variation annuelle par denrée
fact_nat = fact_nat.sort_values(["commodity_fr", "date"])
fact_nat["var_annuelle_pct"] = (fact_nat.groupby("commodity_fr")["prix_median"]
                                .pct_change(12) * 100)
# WB en format large (1 colonne par indicateur)
wb_wide = wb.pivot(index="annee", columns="code", values="valeur").reset_index()
fact_nat.tail(2)


,date,commodity_fr,prix_median,prix_moyen,nb_marches,annee,cpi,prix_reel_2010,var_annuelle_pct
1653,2026-02-01,Sorgho importé,300.0,318.764706,17,2026,134.103095,223.708483,-25.000000
1654,2026-03-01,Sorgho importé,300.0,303.750000,12,2026,134.103095,223.708483,-14.285714


### 8. Dimensions + référentiel géographique des marchés

In [9]:

dim_commodity = (p.groupby(["commodity_fr", "category"]).size().reset_index(name="n_releves")
                   .assign(dans_panier=lambda d: d["commodity_fr"].isin(PANIER)))
dim_region = (p.groupby(["region", "region_fr"])["market"].nunique()
                .reset_index(name="nb_marches"))
# Référentiel marchés (coordonnées GPS réelles)
markets_geo = (p.groupby(["market", "region", "region_fr"])
                 .agg(latitude=("latitude", "first"), longitude=("longitude", "first"),
                      n_releves=("price", "size"),
                      dernier_releve=("date", "max"))
                 .reset_index())
print("Denrées :", len(dim_commodity), "| Régions :", len(dim_region),
      "| Marchés géolocalisés :", len(markets_geo))
markets_geo.head(3)


Denrées : 12 | Régions : 14 | Marchés géolocalisés : 64


,market,region,region_fr,latitude,longitude,n_releves,dernier_releve
0,Bakel,Tambacounda,Tambacounda,14.90,-12.46,949,2026-03-01
1,Bambey,Diourbel,Diourbel,14.70,-16.46,1084,2026-03-01
2,Bignona,Ziguinchor,Ziguinchor,12.81,-16.23,1037,2026-03-01


### 9. Indice panier régional (base 100 = 2015) pour la carte

In [10]:

reg_rows = []
for reg, g in fact_reg[fact_reg["commodity_fr"].isin(PANIER)].groupby("region"):
    pv = g.pivot_table(index="date", columns="commodity_fr", values="prix_median").asfreq("MS").interpolate(limit=3)
    if "2015" not in pv.index.strftime("%Y"):
        continue
    b = pv.loc["2015"].mean()
    if b.isna().all():
        continue
    rl = pv.divide(b, axis=1) * 100
    cols = [c for c in w.index if c in rl.columns]
    idx = (rl[cols] * w[cols]).sum(axis=1) / w[cols].sum()
    tmp = pd.DataFrame({"date": idx.index, "region": reg, "indice_panier": idx.values})
    reg_rows.append(tmp)
panier_reg = pd.concat(reg_rows, ignore_index=True)
panier_reg["region_fr"] = panier_reg["region"].replace(REGION_FR)
panier_reg["var_annuelle_pct"] = (panier_reg.sort_values("date")
                                  .groupby("region")["indice_panier"].pct_change(12) * 100)
print("Indice régional :", panier_reg.shape, "| régions :", panier_reg["region"].nunique())


Indice régional : (3225, 5) | régions : 14


### 10. Écriture du modèle en étoile dans `data/processed/`

In [11]:

tables = {
    "fact_prix_national": fact_nat, "fact_prix_regional": fact_reg,
    "indice_panier_national": panier_nat, "indice_panier_regional": panier_reg,
    "dim_commodity": dim_commodity, "dim_region": dim_region,
    "markets_geo": markets_geo, "worldbank": wb_wide, "inflation_compare": comp.reset_index(),
}
for name, df in tables.items():
    df.to_csv(os.path.join(PROC, f"{name}.csv"), index=False, encoding="utf-8-sig")
    print(f"  {name:24s} {df.shape}")
print("\n✅ Données réelles nettoyées et structurées.")


  fact_prix_national       (1655, 9)


  fact_prix_regional       (18805, 6)
  indice_panier_national   (231, 4)
  indice_panier_regional   (3225, 5)
  dim_commodity            (12, 4)
  dim_region               (14, 3)
  markets_geo              (64, 7)
  worldbank                (66, 9)
  inflation_compare        (17, 3)

✅ Données réelles nettoyées et structurées.
